In [1]:
import os
import cv2
import sys
import json
import time
import joblib
import numpy as np

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model

from skimage.filters import gabor
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.utils import class_weight

from joblib import Parallel, delayed
from multiprocessing import cpu_count


BASE_DATA_DIR   = "/kaggle/input/datasets/medasivamanoj/leaf-disease"
TRAIN_DIR       = os.path.join(BASE_DATA_DIR, "train1",      "train1")
VAL_DIR         = os.path.join(BASE_DATA_DIR, "validation1", "validation1")
TEST_DIR        = os.path.join(BASE_DATA_DIR, "test1",       "test1")
IMG_SIZE_CNN    = (128, 128)
BATCH_SIZE      = 32
CNN_EPOCHS      = 12
NUM_CLASSES     = 38

FREQUENCIES     = [0.1, 0.2, 0.3, 0.4]
THETAS          = [0, np.pi/4, np.pi/2, 3*np.pi/4]



def select_classes(train_dir, n):
    all_classes = sorted([
        d for d in os.listdir(train_dir)
        if os.path.isdir(os.path.join(train_dir, d))
    ])
    return all_classes[:n]


def build_and_train_cnn(selected_classes):
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        TRAIN_DIR,
        image_size=IMG_SIZE_CNN,
        batch_size=BATCH_SIZE,
        shuffle=True,
        class_names=selected_classes
    )

    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        VAL_DIR,
        image_size=IMG_SIZE_CNN,
        batch_size=BATCH_SIZE,
        shuffle=False,
        class_names=selected_classes
    )

    class_names = train_ds.class_names
    with open("class_names.json", "w") as f:
        json.dump(class_names, f)

    y_train_list = []
    for _, y_batch in train_ds:
        y_train_list.extend(y_batch.numpy().tolist())
    y_train_list = np.array(y_train_list)

    weights = class_weight.compute_class_weight(
        'balanced', classes=np.unique(y_train_list), y=y_train_list
    )
    weight_dict = dict(enumerate(weights))

    inputs = layers.Input(shape=(128, 128, 3))
    x = layers.Rescaling(1./255)(inputs)

    x = layers.Conv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.GlobalAveragePooling2D(name="feature_output")(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(class_names), activation='softmax')(x)

    full_model = models.Model(inputs, outputs)
    full_model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=0
    )

    full_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CNN_EPOCHS,
        callbacks=[early_stop],
        class_weight=weight_dict,
        verbose=0
    )

    full_model.save("plant_disease_model.keras")
    return full_model, class_names


def get_gabor_features(path):
    img = cv2.imread(path)
    if img is None:
        return None
    gray = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (128, 128))
    g_feats = []
    for t in THETAS:
        for f in FREQUENCIES:
            filt_real, _ = gabor(gray, frequency=f, theta=t)
            g_feats.extend([
                filt_real.mean(),
                filt_real.var(),
                filt_real.std(),
                np.max(filt_real)
            ])
    return g_feats


def process_directories(dataset_dir, selected_classes):
    paths, labels = [], []
    for label in sorted(os.listdir(dataset_dir)):
        if label not in selected_classes:
            continue
        c_path = os.path.join(dataset_dir, label)
        if not os.path.isdir(c_path):
            continue
        for img_name in os.listdir(c_path):
            paths.append(os.path.join(c_path, img_name))
            labels.append(label)

    features = Parallel(n_jobs=-1)(delayed(get_gabor_features)(p) for p in paths)

    valid_triplets = [
        (f, l, p)
        for f, l, p in zip(features, labels, paths)
        if f is not None
    ]

    feats = np.array([v[0] for v in valid_triplets])
    labs  = np.array([v[1] for v in valid_triplets])
    pths  = [v[2] for v in valid_triplets]

    return feats, labs, pths


def extract_cnn_features(paths, extractor):
    feats = []
    for i in range(0, len(paths), BATCH_SIZE):
        batch_paths = paths[i:i + BATCH_SIZE]
        imgs = []
        for p in batch_paths:
            img = cv2.imread(p)
            if img is None:
                imgs.append(np.zeros((128, 128, 3), dtype=np.float32))
            else:
                imgs.append(
                    cv2.resize(
                        cv2.cvtColor(img, cv2.COLOR_BGR2RGB), IMG_SIZE_CNN
                    ) / 255.0
                )
        feats.append(extractor.predict(np.array(imgs), verbose=0))

    return np.vstack(feats)


if not os.path.exists(TRAIN_DIR):
    raise FileNotFoundError(f"Training directory not found: {TRAIN_DIR}")

if not os.path.exists(VAL_DIR):
    raise FileNotFoundError(f"Validation directory not found: {VAL_DIR}")

selected_classes = select_classes(TRAIN_DIR, NUM_CLASSES)

full_model, class_names = build_and_train_cnn(selected_classes)
extractor = Model(
    inputs=full_model.input,
    outputs=full_model.get_layer("feature_output").output
)

X_gab_train, y_str_train, paths_train = process_directories(TRAIN_DIR, selected_classes)
X_gab_val, y_str_val, paths_val = process_directories(VAL_DIR, selected_classes)

X_cnn_train = extract_cnn_features(paths_train, extractor)
X_cnn_val = extract_cnn_features(paths_val, extractor)

X_fused_train = np.hstack([X_cnn_train, X_gab_train])
X_fused_val   = np.hstack([X_cnn_val,   X_gab_val])

le = LabelEncoder()
y_train = le.fit_transform(y_str_train)
y_val   = le.transform(y_str_val)
joblib.dump(le, "label_encoder.pkl")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_fused_train)
X_val_sc   = scaler.transform(X_fused_val)
joblib.dump(scaler, "scaler_fused.pkl")

estimators = [
    ('rf', RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42,
        verbose=0
    )),
    ('svm', SVC(
        kernel='rbf',
        probability=True,
        class_weight='balanced'
    )),
]

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    n_jobs=-1,
    verbose=0
)


stack.fit(X_train_sc, y_train)
joblib.dump(stack, "stacking_fused.pkl")

final_preds = stack.predict(X_val_sc)
val_report = classification_report(y_val, final_preds, target_names=class_names)

with open("classification_report.txt", "w") as f:
    f.write(val_report)

if os.path.exists(TEST_DIR):
    X_gab_test, y_str_test, paths_test = process_directories(TEST_DIR, selected_classes)
    X_cnn_test = extract_cnn_features(paths_test, extractor)

    X_fused_test = np.hstack([X_cnn_test, X_gab_test])
    X_test_sc = scaler.transform(X_fused_test)
    y_test = le.transform(y_str_test)

    test_preds = stack.predict(X_test_sc)
    test_report = classification_report(y_test, test_preds, target_names=class_names)

    with open("test_classification_report.txt", "w") as f:
        f.write(test_report)

2026-04-27 14:15:26.941750: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777299327.186379      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777299327.255050      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777299327.794037      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777299327.794091      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777299327.794094      16 computation_placer.cc:177] computation placer alr

Found 27494 files belonging to 38 classes.
Found 608 files belonging to 38 classes.


2026-04-27 14:16:03.989223: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
